# Medical Insurance Cost Analysis

This notebook analyses a medical insurance dataset and develops regression models to estimate annual insurance charges.

## Objectives

- Load and clean the dataset.
- Explore relationships between the available variables and insurance charges.
- Develop single-variable and multivariable regression models.
- Refine the model using polynomial features and Ridge regression.
- Evaluate the final model on a held-out test set.
- Produce portfolio-ready visualizations of the exploratory analysis and final predictions.

| Parameter | Description |
|---|---|
| `age` | Age in years |
| `gender` | Gender encoded numerically |
| `bmi` | Body mass index |
| `no_of_children` | Number of children |
| `smoker` | Smoking status (0 = non-smoker, 1 = smoker) |
| `region` | US region encoded numerically |
| `charges` | Annual insurance charges in USD |

## 1. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

%matplotlib inline

## 2. Load the dataset

In [ ]:
filepath = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/medical_insurance_dataset.csv"

headers = [
    "age",
    "gender",
    "bmi",
    "no_of_children",
    "smoker",
    "region",
    "charges",
]

df = pd.read_csv(filepath, header=None, names=headers)
df.head(10)

## 3. Data cleaning

In [ ]:
# Replace missing-value marker with NaN
df.replace("?", np.nan, inplace=True)

# Age is continuous: replace missing values with the mean
mean_age = df["age"].astype(float).mean()
df["age"] = df["age"].fillna(mean_age)

# Smoking status is categorical: replace missing values with the mode
most_common_smoker = df["smoker"].mode()[0]
df["smoker"] = df["smoker"].fillna(most_common_smoker)

# Correct data types
df[["age", "smoker"]] = df[["age", "smoker"]].astype(int)

# Round insurance charges
df["charges"] = df["charges"].round(2)

df.info()

In [ ]:
df.head()

## 4. Exploratory Data Analysis

First, inspect the relationship between BMI and insurance charges.

In [ ]:
plt.figure(figsize=(8, 6))
sns.regplot(x="bmi", y="charges", data=df, line_kws={"color": "red"})
plt.ylim(0)
plt.xlabel("BMI")
plt.ylabel("Annual insurance charges (USD)")
plt.title("Insurance charges vs. BMI")
plt.show()

### Smoking status and insurance charges

Smoking status shows the strongest linear association with insurance charges in this dataset.

The following visualization is formatted so it can also be used directly in the Spanish-language portfolio.

In [ ]:
plt.figure(figsize=(8, 6))

ax = sns.boxplot(x="smoker", y="charges", data=df)

ax.set_xticks([0, 1])
ax.set_xticklabels(["No fumador", "Fumador"])
ax.set_xlabel("Estatus de fumador")
ax.set_ylabel("Coste anual del seguro (USD)")
ax.set_title("Coste anual del seguro según el estatus de fumador")

plt.tight_layout()
plt.show()

### Correlation matrix

In [ ]:
correlation_matrix = df.corr(numeric_only=True)
correlation_matrix

## 5. Model development

### Single-variable linear regression

Use smoking status as the only predictor.

In [ ]:
X = df[["smoker"]]
Y = df["charges"]

lm = LinearRegression()
lm.fit(X, Y)

single_r2 = lm.score(X, Y)
print(f"Single-variable linear regression R²: {single_r2:.3f}")

### Multivariable linear regression

Use all available predictors.

In [ ]:
Z = df[[
    "age",
    "gender",
    "bmi",
    "no_of_children",
    "smoker",
    "region",
]].astype(float)

lm.fit(Z, Y)
multiple_r2 = lm.score(Z, Y)

print(f"Multivariable linear regression R²: {multiple_r2:.3f}")

### Polynomial linear-regression pipeline

This is an **in-sample** score and should not be interpreted as held-out predictive performance.

In [ ]:
pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("polynomial", PolynomialFeatures(include_bias=False)),
    ("model", LinearRegression()),
])

pipeline.fit(Z, Y)
y_pipeline = pipeline.predict(Z)

pipeline_r2 = r2_score(Y, y_pipeline)
print(f"In-sample polynomial pipeline R²: {pipeline_r2:.3f}")

## 6. Model refinement and held-out evaluation

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    Z,
    Y,
    test_size=0.2,
    random_state=1,
)

print(f"Training observations: {len(x_train)}")
print(f"Testing observations: {len(x_test)}")

### Ridge regression

In [ ]:
ridge_model = Ridge(alpha=0.1)
ridge_model.fit(x_train, y_train)

yhat_ridge = ridge_model.predict(x_test)
ridge_r2 = r2_score(y_test, yhat_ridge)

print(f"Ridge test R²: {ridge_r2:.3f}")

### Polynomial features + Ridge regression

Polynomial features are **fitted only on the training set** and then applied to the test set using `transform()`.  
This avoids fitting preprocessing steps independently on held-out data.

In [ ]:
poly = PolynomialFeatures(degree=2)

# Fit the feature transformation only on training data
x_train_poly = poly.fit_transform(x_train)

# IMPORTANT: apply the already-fitted transformation to test data
x_test_poly = poly.transform(x_test)

ridge_poly_model = Ridge(alpha=0.1)
ridge_poly_model.fit(x_train_poly, y_train)

y_hat = ridge_poly_model.predict(x_test_poly)
final_r2 = r2_score(y_test, y_hat)

print(f"Polynomial Ridge test R²: {final_r2:.3f}")

## 7. Final portfolio visualization — actual vs. predicted costs

Each point represents one observation in the held-out test set.

The dashed diagonal represents perfect agreement between the observed insurance cost and the model prediction. Points closer to this line correspond to more accurate predictions.

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_hat, alpha=0.6)

min_value = min(y_test.min(), y_hat.min())
max_value = max(y_test.max(), y_hat.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    label="Predicción perfecta",
)

plt.xlabel("Coste real (USD)")
plt.ylabel("Coste predicho (USD)")
plt.title("Costes reales vs. predichos")

plt.text(
    0.05,
    0.92,
    f"R² test = {final_r2:.3f}",
    transform=plt.gca().transAxes,
)

plt.legend()
plt.tight_layout()
plt.show()

## 8. Summary

The analysis shows that smoking status has the strongest linear association with insurance charges among the variables considered.

The final model combines second-degree polynomial features with Ridge regression and is evaluated on a held-out 20% test subset. The test-set R² should be used when discussing predictive performance, rather than the higher in-sample scores reported earlier in the notebook.

### Portfolio workflow

**Data cleaning → Exploratory analysis → Regression modelling → Held-out evaluation → Prediction visualization**